# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [4]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning

## Base

In [5]:
pipe = make_pipeline(
    RareLabelEncoder(variables=['driver']),
    HistGradientBoostingClassifier(class_weight='balanced', verbose=0),
)

In [6]:
cv_results = cross_val_score(
    estimator=pipe,
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1
)

In [7]:
cv_results.mean()

np.float64(0.9450655943692249)

In [8]:
pipe.fit(X_train, y_train.PitNextLap)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('rarelabelencoder', ...), ('histgradientboostingclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,tol,0.05
,n_categories,10
,max_n_categories,None
,replace_with,'Rare'
,variables,['driver']
,missing_values,'raise'
,ignore_format,False


## Feature Selection

In [9]:
perm_result = permutation_importance(
    estimator=pipe, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1, 
    scoring='roc_auc'
)


importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [10]:
importance_df.query("importance_mean <= 0").feature.tolist()

['driver',
 'raceprogress_high',
 'lapnumber_high',
 'race_progress_squared',
 'position_norm',
 'is_pit_lap',
 'position_high',
 'laptime_s_low',
 'laptime_s_high',
 'treeraceprogress_laptime_s',
 'stint_high',
 'treeposition',
 'treeraceprogress',
 'treelaptime_s',
 'treeraceprogress_position',
 'treeposition_laptime_s',
 'lapnumber_low']

In [11]:
features_to_drop = importance_df.query("importance_mean <= 0").feature.tolist()

## Fine Tuning

In [13]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = make_pipeline(
            DropFeatures(features_to_drop),
            HistGradientBoostingClassifier(
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 10),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 20, 100),
                l2_regularization=trial.suggest_float("l2_regularization", 0.0, 10.0),
                max_bins=trial.suggest_int("max_bins", 64, 255),
                max_iter=1000,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=50,
                random_state=42,
                class_weight='balanced', 
                verbose=0
            )
        ).fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30, n_jobs=-1, show_progress_bar=True)


print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-11 11:11:34,243] A new study created in memory with name: no-name-c76b2654-30c0-4ed2-b050-a8d74fdff5ab
Best trial: 10. Best value: 0.946888:   3%|█▎                                       | 1/30 [03:50<1:51:15, 230.19s/it]

[I 2026-05-11 11:15:24,427] Trial 10 finished with value: 0.9468880929797919 and parameters: {'learning_rate': 0.2445214310923147, 'max_depth': 4, 'min_samples_leaf': 45, 'l2_regularization': 6.016380757145264, 'max_bins': 196}. Best is trial 10 with value: 0.9468880929797919.


Best trial: 8. Best value: 0.947707:   7%|██▊                                       | 2/30 [05:04<1:04:32, 138.29s/it]

[I 2026-05-11 11:16:38,378] Trial 8 finished with value: 0.9477074120048614 and parameters: {'learning_rate': 0.1426505287151401, 'max_depth': 5, 'min_samples_leaf': 41, 'l2_regularization': 3.1156022690449103, 'max_bins': 140}. Best is trial 8 with value: 0.9477074120048614.


Best trial: 8. Best value: 0.947707:  10%|████▍                                       | 3/30 [06:42<54:05, 120.22s/it]

...

[I 2026-05-11 11:35:41,454] Trial 27 finished with value: 0.9483567024498891 and parameters: {'learning_rate': 0.05039263406873434, 'max_depth': 8, 'min_samples_leaf': 100, 'l2_regularization': 9.998610623720424, 'max_bins': 168}. Best is trial 23 with value: 0.9485286479852023.


Best trial: 23. Best value: 0.948529:  97%|█████████████████████████████████████████▌ | 29/30 [24:15<00:17, 17.87s/it]

[I 2026-05-11 11:35:49,641] Trial 28 finished with value: 0.9483380314682617 and parameters: {'learning_rate': 0.05319198115513419, 'max_depth': 8, 'min_samples_leaf': 58, 'l2_regularization': 9.743946244406143, 'max_bins': 250}. Best is trial 23 with value: 0.9485286479852023.


Best trial: 23. Best value: 0.948529: 100%|███████████████████████████████████████████| 30/30 [24:16<00:00, 48.56s/it]

[I 2026-05-11 11:35:50,965] Trial 29 finished with value: 0.948426549289333 and parameters: {'learning_rate': 0.056264072404117044, 'max_depth': 8, 'min_samples_leaf': 61, 'l2_regularization': 9.468606841084034, 'max_bins': 255}. Best is trial 23 with value: 0.9485286479852023.
Best AUC: 0.9485286479852023
Best params: {'learning_rate': 0.05650776345088367, 'max_depth': 8, 'min_samples_leaf': 99, 'l2_regularization': 9.754095966016491, 'max_bins': 65}


In [14]:
pipe_tuned = make_pipeline(
    DropFeatures(features_to_drop),
    HistGradientBoostingClassifier(
        **study.best_params,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=50,
        random_state=42,
        class_weight='balanced',
        verbose=0
    )
).fit(X_train, y_train.PitNextLap)

# Deploy

In [15]:
dump_pickle(pipe_tuned, '../models/model_histgradientboosting.pkl')